# 08 · Measure retrieval quality — the baseline

> **Run order.** This notebook is step 8 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


The number everything after this is measured against.

**Zero LLM calls.** Recall@k and MRR compare retrieved element IDs against the
ground-truth element IDs notebook 06 anchored — pure computation. That matters
practically as well as intellectually: the most iterative days of this project
cost nothing against any provider's rate limit.

A hit means a retrieved chunk contains one of the expected `element_id`s.
`pR@5` is the softer *page*-level recall: did we surface the right page at all?

Every run is **recorded**, not just printed. `analyst.evaluation` appends each one
to `results/runs.jsonl` with the config that produced it, a hash of the benchmark
it scored, and the git revision of the code — so "hybrid beat dense by X" is a
claim you can re-open six weeks later instead of re-running.

In [ ]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.embedding import MODELS
from analyst.retrievers import dense, open_store

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(f"{len(questions)} questions   benchmark {ev.bench_sha(questions)}   git {ev.git_rev()}")
pd.DataFrame([q.model_dump() for q in questions]).groupby(
    ["question_type", "match_kind"]).size().to_frame("n")

## One function, every configuration

The retriever is injected into `ev.evaluate`, so dense, hybrid and reranked
retrieval are scored by identical code on identical questions. That is the only
reason the deltas mean anything.

In [ ]:
def score(model: str, use_filters: bool = True, deep: bool = False) -> ev.Run:
    """Evaluate one config, append it to the ledger, return the record."""
    embedder, store = open_store(settings, model)
    search = dense(embedder, store, use_filters)
    cfg = ev.RunConfig(retriever="dense", model=model,
                       use_filters=use_filters, limit=max(ev.K_VALUES))
    run = ev.build_run(
        cfg,
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
        # A second, wider pass: recall at depth is the ceiling for anything that
        # only reorders results, so it decides whether a reranker can help.
        deep=ev.evaluate(questions, search, limit=max(ev.DEPTHS)) if deep else None,
    )
    ev.append_run(run)
    return run

print("ready")

## Baseline: dense retrieval, metadata filters on

In [ ]:
base = score("bge-small", use_filters=True, deep=True)
pd.DataFrame([base.row()])

## Does metadata filtering earn its complexity?

Filtering restricts the candidate set *before* scoring. The claim is that this is
both faster and more accurate than filtering afterwards. Claims get measured here.

In [ ]:
nofilter = score("bge-small", use_filters=False)
pd.DataFrame([nofilter.row(), base.row()])

## The measurement that decides Day 4

A cross-encoder reranker only reorders what retrieval already surfaced. Wherever
this curve goes flat is its hard ceiling, however good the reranker is.

In [ ]:
pd.DataFrame([base.depth_curve], index=["recall"]).rename_axis("depth", axis=1)

## ADR-006: which embedding model?

Every candidate in `analyst.embedding.MODELS` is indexed as its own collection by
notebook 07 and scored here on the same 44 questions. A model is chosen because it
won a measurement, not because of its reputation.

Collections that have not been indexed are skipped rather than failing the notebook.

In [ ]:
for model in MODELS:
    if model == "bge-small":
        continue
    try:
        score(model, use_filters=True, deep=True)
    except Exception as exc:  # collection not indexed yet
        print(f"skip {model}: {type(exc).__name__}")

runs = ev.load_runs()
pd.DataFrame([r.row() for r in runs]).sort_values("R@5", ascending=False)

## Where does it fail?

Aggregate numbers hide the interesting part.

In [ ]:
worst = ev.evaluate(questions, dense(*open_store(settings, base.config.model)),
                    limit=max(ev.K_VALUES))
df = pd.DataFrame([r.model_dump() for r in worst])
print(df.groupby("question_type")["rank"].agg(
    n="size", found="count", best="min").to_string())

print(f"\nNever retrieved in the top {max(ev.K_VALUES)}: "
      f"{df['rank'].isna().sum()} of {len(df)}")
df[df["rank"].isna()].groupby(["ticker", "question_type"]).size().to_frame("misses")

## The ledger

`results/leaderboard.md` is regenerated from `results/runs.jsonl` — committed, and
never edited by hand. Every future retrieval change appends to the same file, so
the delta is always one table away.

In [ ]:
print(ev.write_leaderboard(ev.load_runs()))
print(ev.render_leaderboard(ev.load_runs()))